# CIFAR-10 Training Lab

This notebook is the interactive lab for the neural-network training chapter. It wraps the existing PyTorch runner instead of reimplementing the model, so the command-line scripts remain the source of truth for reproducible runs and saved artifacts.

Use the notebook to inspect the workflow, run small controlled comparisons, and collect the history, curve, confusion-matrix, per-class-accuracy, and metadata files needed for a short report.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## Setup

Run this notebook from a Python environment with the shared code dependencies installed:

```sh
poetry install --with pytorch,figures
```

The first executable cell locates the repository root, the existing PyTorch runner, and the notebook artifact directory.

In [ ]:
from pathlib import Path
import csv
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        runner = path / "chapter_cnn_architectures" / "resnet50_cifar10_pytorch.py"
        if (path / "pyproject.toml").exists() and runner.exists():
            return path
        legacy_runner = path / "code" / "chapter_cnn_architectures" / "resnet50_cifar10_pytorch.py"
        if (path / "code" / "pyproject.toml").exists() and legacy_runner.exists():
            return path / "code"
    raise RuntimeError("Could not find the companion-code repository root from this notebook.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = REPO_ROOT
RUNNER_DIR = REPO_ROOT / "chapter_cnn_architectures"
RUNNER = RUNNER_DIR / "resnet50_cifar10_pytorch.py"
ARTIFACT_DIR = REPO_ROOT / "chapter_neural_network_training" / "runs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Runner:     {RUNNER}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Python:     {sys.executable}")

## Command Helpers

The helper below builds the same command you would run in a terminal. Each training run writes files named from `run_id`, so use stable run IDs for controlled comparisons.

In [ ]:
def shell_join(command: list[object]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def run_command(command: list[object], *, cwd: Path = RUNNER_DIR, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", shell_join(command))
    result = subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def training_command(run_id: str, **overrides: object) -> list[object]:
    options: dict[str, object] = {
        "epochs": 2,
        "batch_size": 128,
        "seed": 1234,
        "validation_size": 5000,
        "num_workers": 0,
        "limit_train": 512,
        "limit_val": 256,
        "limit_test": None,
        "stem": "cifar",
        "model_variant": "resnet18",
        "optimizer": "sgd",
        "learning_rate": 0.05,
        "momentum": 0.0,
        "weight_decay": 5e-4,
        "schedule": "constant",
        "warmup_epochs": 0,
        "normalization": "batchnorm",
        "dropout_rate": 0.0,
        "augmentation": "basic",
        "synthetic_data": False,
        "quick": False,
        "evaluate_test": False,
    }
    options.update(overrides)

    command: list[object] = [
        sys.executable,
        RUNNER,
        "--run-id",
        run_id,
        "--epochs",
        options["epochs"],
        "--batch-size",
        options["batch_size"],
        "--seed",
        options["seed"],
        "--validation-size",
        options["validation_size"],
        "--num-workers",
        options["num_workers"],
        "--stem",
        options["stem"],
        "--model-variant",
        options["model_variant"],
        "--optimizer",
        options["optimizer"],
        "--learning-rate",
        options["learning_rate"],
        "--momentum",
        options["momentum"],
        "--weight-decay",
        options["weight_decay"],
        "--schedule",
        options["schedule"],
        "--warmup-epochs",
        options["warmup_epochs"],
        "--normalization",
        options["normalization"],
        "--dropout-rate",
        options["dropout_rate"],
        "--augmentation",
        options["augmentation"],
        "--figure-dir",
        ARTIFACT_DIR,
        "--save-figures",
    ]

    for option_name, cli_name in [
        ("limit_train", "--limit-train"),
        ("limit_val", "--limit-val"),
        ("limit_test", "--limit-test"),
    ]:
        if options[option_name] is not None:
            command.extend([cli_name, options[option_name]])

    if options["synthetic_data"]:
        command.append("--synthetic-data")
    if options["quick"]:
        command.append("--quick")
    if options["evaluate_test"]:
        command.append("--evaluate-test")
    return command


def run_training(run_id: str, **overrides: object) -> subprocess.CompletedProcess[str]:
    return run_command(training_command(run_id, **overrides))

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## Artifact Helpers

These functions load the files produced by the runner and display the same evidence the chapter asks you to report.

In [ ]:
def artifact_path(run_id: str, suffix: str) -> Path:
    return ARTIFACT_DIR / f"{run_id}-{suffix}"


def read_history(run_id: str) -> list[dict[str, float]]:
    path = artifact_path(run_id, "history.csv")
    with path.open(newline="", encoding="utf-8") as csv_file:
        rows = []
        for row in csv.DictReader(csv_file):
            rows.append({key: float(value) for key, value in row.items()})
    return rows


def read_metadata(run_id: str) -> dict[str, str]:
    path = artifact_path(run_id, "metadata.txt")
    metadata: dict[str, str] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if ": " in line and not line.startswith("- "):
            key, value = line.split(": ", 1)
            metadata[key] = value
    return metadata


def show_history(run_id: str) -> None:
    rows = read_history(run_id)
    lines = ["| epoch | train loss | train acc | val loss | val acc | lr |", "|---:|---:|---:|---:|---:|---:|"]
    for row in rows:
        lines.append(
            f"| {int(row['epoch'])} | {row['train_loss']:.4f} | {row['train_accuracy']:.4f} | "
            f"{row['val_loss']:.4f} | {row['val_accuracy']:.4f} | {row['learning_rate']:.6g} |"
        )
    display(Markdown("\n".join(lines)))


def show_artifacts(run_id: str) -> None:
    print(f"Artifacts for {run_id}: {ARTIFACT_DIR}")
    for suffix in ["training-curves.png", "confusion-matrix.png"]:
        path = artifact_path(run_id, suffix)
        if path.exists():
            display(Image(filename=str(path)))
        else:
            print(f"Missing {path.name}; run the training cell first.")
    history = artifact_path(run_id, "history.csv")
    if history.exists():
        show_history(run_id)


def summarize_runs(run_ids: list[str]) -> None:
    lines = [
        "| run | best val acc | final val acc | test acc | elapsed seconds | optimizer | schedule | weight decay | dropout |",
        "|---|---:|---:|---:|---:|---|---|---:|---:|",
    ]
    for run_id in run_ids:
        metadata_path = artifact_path(run_id, "metadata.txt")
        if not metadata_path.exists():
            lines.append(f"| {run_id} | missing | missing | missing | missing | | | | |")
            continue
        metadata = read_metadata(run_id)
        lines.append(
            f"| {run_id} | {metadata.get('Best validation accuracy', '')} | "
            f"{metadata.get('Final validation accuracy', '')} | {metadata.get('Test accuracy', 'skipped')} | "
            f"{metadata.get('Elapsed seconds', '')} | {metadata.get('Optimizer', '')} | "
            f"{metadata.get('Schedule', '')} | {metadata.get('Weight decay', '')} | "
            f"{metadata.get('Dropout rate', '')} |"
        )
    display(Markdown("\n".join(lines)))

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## Dependency Check and Synthetic Smoke Run

The smoke run uses synthetic CIFAR-shaped data. It checks that the training loop, model, optimizer, artifact writing, and image display path work before you spend time on real CIFAR-10 experiments.

In [ ]:
run_command([sys.executable, RUNNER, "--check-deps", "--allow-missing-deps", "--synthetic-data"])

After the dependency report, run one tiny synthetic training pass. This confirms that the runner can train, save artifacts, and display evidence before you spend time on controlled CIFAR-10 runs.

In [ ]:
run_training("nnt-notebook-smoke", quick=True)
show_artifacts("nnt-notebook-smoke")

## Choose an Experiment Budget

The default budget below keeps the notebook responsive. To mirror the bounded chapter artifact table more closely, change `limit_train` to `4096`, `limit_val` to `1024`, and `epochs` to `3`. For the capstone homework, remove the limits and use a longer epoch budget chosen for your hardware.

In [ ]:
RUN_BUDGET = {
    "epochs": 2,
    "batch_size": 128,
    "validation_size": 5000,
    "limit_train": 512,
    "limit_val": 256,
    "limit_test": 256,
    "model_variant": "resnet18",
    "stem": "cifar",
    "augmentation": "basic",
    "synthetic_data": False,
}

RUN_BUDGET

## Controlled Run 1: Plain Baseline

Start with one complete recipe: fixed split, architecture, optimizer, schedule, augmentation, and artifact directory. This baseline uses SGD without momentum and a constant learning rate.

In [ ]:
run_training(
    "nnt-lab-sgd-baseline",
    **RUN_BUDGET,
    optimizer="sgd",
    learning_rate=0.05,
    momentum=0.0,
    weight_decay=5e-4,
    schedule="constant",
    warmup_epochs=0,
    dropout_rate=0.0,
)
show_artifacts("nnt-lab-sgd-baseline")

## Controlled Run 2: Optimizer and Schedule Change

Change the optimizer and schedule while keeping the data split, model, augmentation, and regularization surface fixed. The warmup value must be smaller than the epoch count.

In [ ]:
run_training(
    "nnt-lab-adamw-cosine",
    **RUN_BUDGET,
    optimizer="adamw",
    learning_rate=0.001,
    momentum=0.9,
    weight_decay=5e-4,
    schedule="cosine",
    warmup_epochs=1,
    dropout_rate=0.0,
)
show_artifacts("nnt-lab-adamw-cosine")

## Controlled Run 3: Regularization Change

Now keep the AdamW plus cosine recipe and add classifier-head dropout. Compare validation behavior rather than assuming the regularizer helps.

In [ ]:
run_training(
    "nnt-lab-adamw-cosine-dropout",
    **RUN_BUDGET,
    optimizer="adamw",
    learning_rate=0.001,
    momentum=0.9,
    weight_decay=5e-4,
    schedule="cosine",
    warmup_epochs=1,
    dropout_rate=0.2,
)
show_artifacts("nnt-lab-adamw-cosine-dropout")

## Compare Runs

Use this table as the starting point for the report. A claim such as "dropout helped" or "AdamW improved early progress" needs support from this table and the curves above.

In [ ]:
CONTROLLED_RUNS = [
    "nnt-lab-sgd-baseline",
    "nnt-lab-adamw-cosine",
    "nnt-lab-adamw-cosine-dropout",
]

summarize_runs(CONTROLLED_RUNS)

## Final Selected Test Run

Leave this cell disabled until you have selected a recipe using validation evidence. When you enable it, keep the run ID distinct and include `evaluate_test=True` so the metadata records the one-time test result.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    run_training(
        "nnt-lab-final-selected",
        **RUN_BUDGET,
        optimizer="adamw",
        learning_rate=0.001,
        momentum=0.9,
        weight_decay=5e-4,
        schedule="cosine",
        warmup_epochs=1,
        dropout_rate=0.0,
        evaluate_test=True,
    )
    show_artifacts("nnt-lab-final-selected")
else:
    print("Final test run is disabled. Set RUN_FINAL_TEST = True only after selecting a recipe by validation evidence.")

## Report Checklist

- Fixed split rule and seed
- Configuration table for every controlled run
- Training and validation curves
- Best and final validation accuracy
- One-time test accuracy for the selected model only
- Runtime, framework versions, and device context from metadata
- Short error analysis from the confusion matrix or per-class accuracy CSV
- Honest note about hardware limits, short budgets, or failed runs

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.